# Restaurant assistant agent demo
The restaurant assistant agent brings together number for Amazon Bedrock Agentcore capabilities such as Runtime, Short-term memory and Long-term memory to demonstrate integrated Observability capabilities. The demo uses Strands framework for constructing the AI agent.

### Tutorial details
| Information         | Details                                                  |
|---------------------|----------------------------------------------------------|
| tutorial type       | conversational                                           |
| Agent type          | Single                                                   |
| tutorial components | Strands, Runtime, memory, observability                  |
| tutorial vertical   | General                                                  |
| Example complexity  | Intermediate                                             |
| SDK used            | Amazon Bedrock AgentCore SDK, boto3, Strands             |

You'll learn to:
- Configure AgentCore Memory with extraction strategies for long-term retention
- Hydrate memory with previous conversation history
- Use long-term memory to deliver personalized experiences across conversation sessions
- Integrate Strands Agent Framework with the AgentCore Memory tool

### Scenario Context

In this tutorial, you'll step into the role of a Culinary Assistant designed to deliver highly personalized restaurant recommendations. By leveraging AgentCore Memory's long-term retention and automatic information extraction, the agent can remember user preferences—such as dietary choices and favorite cuisines—across multiple conversations. This persistent memory enables the agent to provide tailored suggestions and a seamless user experience, even as conversations span days or weeks. The scenario demonstrates how structured memory organization and configurable strategies empower conversational AI to move beyond short-term recall, creating truly engaging and context-aware interactions.


### Architecture

<div style="text-align:left">
    <img src="architecture.png" width="65%" />
</div>

### Prerequisites
* Python 3.10+
* UV package manager
* AWS credentials
* Amazon Bedrock model access

#### Before starting initialize Python virtual environment
```
uv venv --python 3.13
source .venv/bin/activate
python -m ensurepip --default-pip
```

In [ ]:
%pip install -r requirements-dev.txt --quiet
%pip install -r requirements.txt --quiet

## Tutorial step-by-step

#### Update environment variables and generate `.env` file

In [ ]:
%%writefile .env
# =============================================================================
# AWS configurations
# =============================================================================
AWS_REGION="us-east-1"

# =============================================================================
# Amazon Bedrock configurations
# =============================================================================
BEDROCK_MODEL_ID = us.anthropic.claude-sonnet-4-5-20250929-v1:0


#### Create Strands agent
The following strands agent leverages AgentCore memory to recall user preferences before generating a response.

In [ ]:
%%writefile strands_agent.py
"""Strands Culinary Assistant"""
import os
import uuid
from ddgs import DDGS
from strands import Agent, tool
from strands.models import BedrockModel
from strands.telemetry import StrandsTelemetry
from strands_tools.agent_core_memory import AgentCoreMemoryToolProvider
# from bedrock_agentcore.memory.integrations.strands.config import (
#     AgentCoreMemoryConfig,
#     RetrievalConfig,
# )
# from bedrock_agentcore.memory.integrations.strands.session_manager import (
#     AgentCoreMemorySessionManager,
# )
from bedrock_agentcore.runtime import BedrockAgentCoreApp, RequestContext
from opentelemetry import baggage, context as trace_context
from opentelemetry.trace import get_tracer_provider
from opentelemetry.processor.baggage import BaggageSpanProcessor, ALLOW_ALL_BAGGAGE_KEYS
from typing import Optional


SYSTEM_PROMPT = """
You are the Culinary Assistant, a sophisticated restaurant recommendation assistant.
PURPOSE:
- Help users discover restaurants based on their preferences
"""
MODEL_ID = os.getenv("BEDROCK_MODEL_ID")
REGION = os.getenv("AWS_REGION", "us-east-1")

# This processor copies all baggage items to span attributes
tracer_provider = get_tracer_provider()
tracer_provider.add_span_processor(BaggageSpanProcessor(ALLOW_ALL_BAGGAGE_KEYS))

# Initialize Strands telemetry for 3P
if os.getenv("DISABLE_ADOT_OBSERVABILITY"):
    strands_telemetry = StrandsTelemetry()
    strands_telemetry.setup_otlp_exporter()

# Initialize agent application wrapper
app = BedrockAgentCoreApp()


def initialize_agent(request_ctx: RequestContext):
    """Initialize the agent with memory tools"""

    model = BedrockModel(
        model_id=MODEL_ID,
    )

    agent = Agent(
        model=model,
        system_prompt=SYSTEM_PROMPT
    )

    return agent


@app.entrypoint
def strands_agent_bedrock(payload, context: Optional[RequestContext] = None):
    """
    Invoke the agent with a payload
    """

    user_input = payload.get("prompt")
    print("User input:", user_input)

    agent = initialize_agent(context)
    response = agent(user_input)
    return response.message['content'][0]['text']


if __name__ == "__main__":
    app.run()


#### Configure AgentCore Runtime for deployment

Next we will use our starter toolkit to configure the AgentCore Runtime deployment with an entrypoint, the execution role we just created and a requirements file. We will also configure the starter kit to auto create the Amazon ECR repository on launch.

During the configure step, your docker file will be generated based on your application code. 

Please note that when using the `bedrock_agentcore_starter_toolkit` to configure your agent, it takes care of the opentelemetry instrumentation. 

In [ ]:
from dotenv import dotenv_values
from bedrock_agentcore_starter_toolkit import Runtime

config = dotenv_values(".env")
region = config.get("AWS_REGION", "us-east-1")

agentcore_runtime = Runtime()
agent_name = "acr_cmk_demo"
response = agentcore_runtime.configure(
    entrypoint="strands_agent.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=agent_name,
    memory_mode='NO_MEMORY',
    # disable_otel=True
)
response

### Deploy to AgentCore Runtime

Now that we've got a docker file, let's launch the agent to the AgentCore Runtime. This will create the Amazon ECR repository and the AgentCore Runtime

In [ ]:
import base64
from dotenv import dotenv_values
from bedrock_agentcore_starter_toolkit import Runtime

config = dotenv_values(".env")

# Bedrock AgentCore configuration
region = config.get("AWS_REGION", "us-east-1")
model_id = config.get("BEDROCK_MODEL_ID", "us.anthropic.claude-sonnet-4-5-20250929-v1:0")

# Langfuse configuration
# otel_endpoint = config.get("LANGFUSE_OTEL_ENDPOINT", "https://us.cloud.langfuse.com/api/public/otel")
# langfuse_secret_key = config.get("LANGFUSE_SECRET_KEY", "")  # For production: key should be securely stored
# langfuse_public_key = config.get("LANGFUSE_PUBLIC_KEY", "")  # For production: key should be securely stored
# langfuse_auth_token = base64.b64encode(f"{langfuse_public_key}:{langfuse_secret_key}".encode()).decode()
# otel_auth_header = f"Authorization=Basic {langfuse_auth_token}"

# Braintrust configuration
# otel_endpoint = config.get("BRAINTRUST_OTEL_ENDPOINT", "https://api.braintrust.dev/otel")
# braintrust_api_key = config.get("BRAINTRUST_API_KEY", "")  # For production: key should be securely stored
# braintrust_project_id = config.get("BRAINTRUST_PROJECT_ID", "")
# otel_auth_header = f"Authorization=Bearer {braintrust_api_key}, x-bt-parent=project_id:{braintrust_project_id}"

launch_result = agentcore_runtime.launch(
    auto_update_on_conflict=True,
    env_vars={
        "BEDROCK_MODEL_ID": model_id,
        "AWS_REGION": region,
        # "DISABLE_ADOT_OBSERVABILITY": "true",
        # "OTEL_EXPORTER_OTLP_ENDPOINT": otel_endpoint,  # Use 3P OTEL endpoint
        # "OTEL_EXPORTER_OTLP_HEADERS": otel_auth_header,  # Add 3P OTEL auth header
    }
)
launch_result

### Check Deployment Status

Wait for the runtime and memory to be ready before invoking:

In [ ]:
import time
status_response = agentcore_runtime.status()
status = status_response.endpoint['status']
end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']
while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']
    print(status)
status

### Invoking AgentCore Runtime

Finally, we can invoke our AgentCore Runtime with a payload

In [ ]:
!agentcore invoke '{"prompt": "suggest sushi restaurants in nyc"}'


## Cleanup instructions

Don't forget to provide the cleanup instructions for any resources created

In [ ]:
!agentcore destroy --delete-ecr-repo --force --dry-run